# Mini-projet : Intégration de l’IA MCP + Agents dans Gemini

Ce notebook met en œuvre une application multi-agents orchestrant plusieurs serveurs MCP (Model Context Protocol).

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0"

## Vérification de l'environnement
Nous vérifions si Node.js et NPM sont disponibles, car ils sont nécessaires pour exécuter certains serveurs MCP via `npx`.

In [ ]:
import subprocess

try:
    node_version = subprocess.check_output(["node", "--version"]).decode().strip()
    npx_version = subprocess.check_output(["npx", "--version"]).decode().strip()
    print(f"Node: {node_version}")
    print(f"Npx: {npx_version}")
except Exception:
    print("Node.js non trouvé. Installation...")
    !apt-get -qq update
    !apt-get -qq install -y nodejs npm
    !node --version
    !npx --version

## Configuration de la clé API Gemini
Assurez-vous d'avoir ajouté votre clé sous le nom `GOOGLE_API_KEY` dans l'onglet "Secrets" (icône clé 🔑) à gauche.

In [ ]:
import os
from google.colab import userdata
import nest_asyncio

# Autoriser les boucles d'événements imbriquées (nécessaire pour Colab + MCP)
nest_asyncio.apply()

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## Création du serveur MCP personnalisé
Nous utilisons `FastMCP` pour créer un serveur local qui expose des outils personnalisés.

In [ ]:
from pathlib import Path
import textwrap

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name="custom_ops")

    @mcp.tool
    def ping() -> str:
        """Outil de vérification de santé."""
        return "pong"

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        """Exemple : compte les lignes d'un texte."""
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}

    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print(f"Serveur personnalisé écrit dans : {server_path}")

## Connexion aux serveurs MCP
Nous utilisons `MultiServerMCPClient` pour agréger les outils provenant de plusieurs serveurs (système de fichiers, Git et notre serveur personnalisé).

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

WORKDIR = "/content"

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    }
}

# Initialisation du client
client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

async def get_tools():
    return await client.get_tools()

# Récupération des outils
tools = asyncio.run(get_tools())

print(f"Nombre total d'outils récupérés : {len(tools)}")
print("Liste des outils :", [t.name for t in tools])

## Initialisation de l'Agent Gemini
Maintenant que nous avons les outils, nous créons un agent capable de les utiliser de manière autonome.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Configuration du modèle LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Création de l'agent ReAct
agent_executor = create_react_agent(llm, tools)

print("L'agent Gemini est prêt avec ses outils MCP !")

## Test de l'Agent Multi-MCP
Nous allons maintenant demander à l'agent d'effectuer une tâche complexe qui nécessite l'utilisation de plusieurs serveurs MCP : créer un fichier via le système de fichiers, puis analyser ses lignes via notre outil personnalisé.

In [ ]:
# Exemple de requête complexe pour l'agent
query = "Crée un fichier nommé 'test_mcp.txt' avec 5 lignes de texte aléatoire. Ensuite, utilise l'outil de résumé pour me dire combien de lignes il contient."

inputs = {"messages": [("user", query)]}

async def run_agent():
    async for event in agent_executor.astream(inputs, stream_mode="values"):
        message = event["messages"][-1]
        message.pretty_print()

asyncio.run(run_agent())